# Notebook 06: Anatomical Body-Region Ablation Study (CPU)

**Objective**: Measure the individual kinematic contribution of each body region by systematically removing its sensor channels while keeping all other conditions identical.

Conditions tested:
1. **All Sensors** (Baseline, 133 channels)
2. **Without Trunk** (114 channels kept)
3. **Without Right Arm** (95 channels kept)
4. **Without Left Arm** (95 channels kept)
5. **Without Right Leg** (111 channels kept)
6. **Without Left Leg** (117 channels kept)

**Compute Environment**: CPU Safe.

## 0. Environment Setup & Repository Bootstrap

In [ ]:
import sys
import os
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    REPO_DIR = Path("/content/Opportunity_HAR")
    if not REPO_DIR.exists():
        print("Cloning Opportunity_HAR from GitHub...")
        !git clone https://github.com/DhruvikaRajput/Opportunity_HAR.git {REPO_DIR}
    if str(REPO_DIR) not in sys.path:
        sys.path.insert(0, str(REPO_DIR))
    os.chdir(REPO_DIR)
    !pip install -q -r requirements.txt
else:
    current = Path.cwd()
    REPO_DIR = current.parent if current.name == "notebooks" else current
    if str(REPO_DIR) not in sys.path:
        sys.path.insert(0, str(REPO_DIR))
    os.chdir(REPO_DIR)

import src.api as api
env_info = api.setup_project()
print(f"Project Initialized on: {env_info['device_name']} (CUDA: {env_info['cuda_available']})")

## 1. Execute Automated Body-Region Ablation
Trains Random Forest under each regional ablation condition using consistent seeds and identical train/test splits.

In [ ]:
try:
    summary_df = api.run_ablation(model_type="random_forest")

    print("\n" + "=" * 75)
    print(" ANATOMICAL BODY-REGION ABLATION RESULTS")
    print("=" * 75)
    print(summary_df[["Ablation Condition", "Channels", "Accuracy", "Macro F1", "Weighted F1"]].to_string(index=False))
    print("=" * 75)
except Exception as e:
    print("Ablation notice:", e)
    print("Ensure preprocessed dataset is available (Notebook 02).")

## 2. Visualize Macro F1 Sensitivity across Body Regions

In [ ]:
try:
    import matplotlib.pyplot as plt
    import seaborn as sns

    plt.figure(figsize=(9, 4))
    ax = sns.barplot(
        data=summary_df,
        x="Ablation Condition",
        y="raw_macro_f1",
        palette=["#2b5c8f" if "All" in c else "#7ea8be" for c in summary_df["Ablation Condition"]]
    )
    baseline_f1 = summary_df.loc[summary_df["Ablation Condition"].str.contains("All"), "raw_macro_f1"].values[0]
    plt.axhline(baseline_f1, color="red", linestyle="--", label=f"All Sensors Baseline ({baseline_f1*100:.2f}%)")
    plt.xticks(rotation=20, ha="right")
    plt.ylabel("Macro F1 Score")
    plt.title("Locomotion Recognition Performance Under Anatomical Region Removal")
    plt.legend()
    plt.tight_layout()
    plt.show()
except Exception as e:
    print("Plotting notice:", e)